# Notebook 02 — Shared Text Preprocessing
**Week 2**: Clean discharge-note text, build TF-IDF features (for Model A) and prepare tokenizer inputs (for Model B). Outputs re-usable label binarizer and feature matrices saved to Drive.

In [ ]:
!pip install -q pandas pyarrow scikit-learn transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Config

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import pickle

DATA_DIR = '/content/drive/MyDrive/mimic_icd/datasets'

# TF-IDF config
TFIDF_MAX_FEATURES = 150_000
TFIDF_NGRAM_RANGE  = (1, 2)
TFIDF_SUBLINEAR_TF = True

# Transformer tokenizer (for Model B reference)
TRANSFORMER_MODEL = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_SEQ_LEN       = 512   # will be handled per-model in notebook 04

## 2. Load cohort splits

In [ ]:
train_df = pd.read_parquet(f'{DATA_DIR}/cohort_train.parquet')
val_df   = pd.read_parquet(f'{DATA_DIR}/cohort_val.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}/cohort_test.parquet')
vocab    = pd.read_csv(f'{DATA_DIR}/label_vocab.csv')['icd_code'].tolist()

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print(f'Label vocab size: {len(vocab)}')

# Restore list from pipe-separated string
for df_ in [train_df, val_df, test_df]:
    df_['icd_codes'] = df_['icd_codes_str'].str.split('|')

## 3. Text cleaning

In [ ]:
# De-identification placeholder patterns common in MIMIC
_DEID_RE = re.compile(
    r'\[\*\*[^\]]*\*\*\]'   # [** ... **]  placeholders
)
_WHITESPACE_RE = re.compile(r'[\s\n\r\t]+')

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = _DEID_RE.sub(' ', text)          # remove de-id tokens
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s.,;:\-/]', ' ', text)  # keep alphanumeric + basic punctuation
    text = _WHITESPACE_RE.sub(' ', text).strip()
    return text

for df_ in [train_df, val_df, test_df]:
    df_['clean_text'] = df_['text'].apply(clean_text)

print('Sample cleaned note (first 300 chars):')
print(train_df['clean_text'].iloc[0][:300])

## 4. Label binarization (MultiLabelBinarizer)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(classes=vocab)

# Fit on vocab (already fixed)
mlb.fit([vocab])

# Transform: only codes in vocab are kept, out-of-vocab codes silently dropped
Y_train = mlb.transform(train_df['icd_codes']).astype(np.float32)
Y_val   = mlb.transform(val_df['icd_codes']).astype(np.float32)
Y_test  = mlb.transform(test_df['icd_codes']).astype(np.float32)

print(f'Y_train shape: {Y_train.shape}   (notes × labels)')
print(f'Label density (train): {Y_train.mean():.4f}')

# Save binarizer
with open(f'{DATA_DIR}/mlb.pkl', 'wb') as f:
    pickle.dump(mlb, f)
print('Saved mlb.pkl')

## 5. TF-IDF vectorisation (Model A features)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp

vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=TFIDF_NGRAM_RANGE,
    sublinear_tf=TFIDF_SUBLINEAR_TF,
    min_df=3,
    strip_accents='unicode',
    analyzer='word',
)

print('Fitting TF-IDF on train set...')
X_train_tfidf = vectorizer.fit_transform(train_df['clean_text'])
X_val_tfidf   = vectorizer.transform(val_df['clean_text'])
X_test_tfidf  = vectorizer.transform(test_df['clean_text'])

print(f'X_train_tfidf shape: {X_train_tfidf.shape}')

# Save
sp.save_npz(f'{DATA_DIR}/X_train_tfidf.npz', X_train_tfidf)
sp.save_npz(f'{DATA_DIR}/X_val_tfidf.npz',   X_val_tfidf)
sp.save_npz(f'{DATA_DIR}/X_test_tfidf.npz',  X_test_tfidf)
np.save(f'{DATA_DIR}/Y_train.npy', Y_train)
np.save(f'{DATA_DIR}/Y_val.npy',   Y_val)
np.save(f'{DATA_DIR}/Y_test.npy',  Y_test)

with open(f'{DATA_DIR}/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print('All features and labels saved.')

## 6. Save cleaned text splits (for transformer notebook)

In [ ]:
for name, df_ in [('train', train_df), ('val', val_df), ('test', test_df)]:
    df_[['subject_id', 'hadm_id', 'clean_text', 'icd_codes_str', 'split']].to_parquet(
        f'{DATA_DIR}/cohort_{name}_clean.parquet', index=False)
print('Clean text parquets saved.')